# Notebook 5 — Causalità di Granger
**Tesi**: Stance detection sui commenti YouTube come predittore del referendum 2026

Questo notebook alimenta il **Capitolo 6** della tesi (RQ3 — causalità di Granger fra stampa, governo e stance referendaria YouTube).

| Sezione | Cosa fa | Riferimento tesi |
|---------|---------|------------------|
| **§1** Setup | Carica `serie_giornaliere_master.csv` o ricostruisci dalle raw | Cap 3, §3.5.1 |
| **§2** Pre-test ADF | Stazionarietà delle 3 serie giornaliere grezze | Cap 3, §3.5.2 — Cap 6, §6.1 |
| **§3** Granger 6 coppie × 2 lag | Test su tutte le direzioni, lag 3 e 7 | Cap 3, §3.5.4 — Cap 6, §6.2 |
| **§4** Heatmap p-value | Visualizzazione matrice causa × effetto | Cap 6, §6.2 |
| **§5** Schema causale | Network diagram dell'asimmetria | Cap 6, §6.3 |
| **§6** Correlazione Pearson | Controllo aggiuntivo (correlazione ≠ causalità) | Cap 6, complemento |
| **§7** Verifica H3 | Schema atteso vs osservato | Cap 6, §6.4 |
| **§8** Termometro vs termostato | Sintesi epistemologica | Cap 6, §6.4.4 |

**Ipotesi H3** (Cap 1):
- Stampa → Referendum significativo a lag positivo
- Governo → Referendum significativo a lag breve
- Direzioni inverse non significative

**Risultati attesi (Cap 6)**: Stampa→Ref a lag 3 (p=0,037) e lag 7 (p=0,0029); Governo→Ref a lag 3 (p=0,038); inversi e laterali non significativi.

**Input atteso da notebook 04:** `serie_giornaliere_master.csv`. Se non disponibile, viene rigenerato al volo dalle stesse fonti.

## §1 — Setup e caricamento serie giornaliere

Tento prima di caricare il master già prodotto dal notebook 04. Se non disponibile,
ricostruisco le 3 serie da raw (stessa logica del notebook 04 §1).

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

WORKING_DIR = "/kaggle/working/"
MASTER_PATH = os.path.join(WORKING_DIR, "serie_giornaliere_master.csv")

PATH_REFERENDUM = "/kaggle/input/datasets/francescochiarolanza/commenti-youtube-referendum-giustizia/risultati_inferenza_completa (2).csv"
PATH_GOVERNO    = "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/inferenza_definitiva.csv"
PATH_STAMPA     = "/kaggle/input/datasets/francescochiarolanza/titoli-giornali-referendum-giustizia/dataset_articoli_referendum.csv"


def serie_youtube(path, nome, col_stance="PREDIZIONE_STANCE", col_data="Data_Commento"):
    """Costruisce serie Sì% giornaliera deduplicata per (Autore, Giorno)."""
    try:
        df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
        if len(df.columns) <= 3:
            df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                             engine="python", on_bad_lines="skip")
    except Exception:
        df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
    df["Data"] = pd.to_datetime(df[col_data], errors="coerce", dayfirst=True, utc=True)
    df = df.dropna(subset=["Data"]).copy()
    df["Data"] = df["Data"].dt.tz_localize(None)
    df["STANCE"] = df[col_stance].astype(str).str.strip().str.upper()
    df = df[df["STANCE"].isin(["POS", "NEG"])]
    df["Giorno"] = df["Data"].dt.date
    if "Autore" in df.columns:
        df = df.sort_values("Data").drop_duplicates(subset=["Autore", "Giorno"], keep="last")
    daily = df.groupby(["Giorno", "STANCE"]).size().unstack(fill_value=0)
    for c in ["POS", "NEG"]:
        if c not in daily.columns:
            daily[c] = 0
    daily["TOT"] = daily["POS"] + daily["NEG"]
    daily = daily[daily["TOT"] >= 5]
    daily.index = pd.to_datetime(daily.index)
    return daily["POS"] / daily["TOT"] * 100


def serie_stampa(path):
    try:
        df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
        if len(df.columns) <= 3:
            df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                             engine="python", on_bad_lines="skip")
    except Exception:
        df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
    date_col   = next((c for c in df.columns if "data" in c.lower()), None)
    stance_col = next((c for c in df.columns if "stance" in c.lower()), None)
    df["Data"] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=["Data"]).copy()
    df["STANCE"] = df[stance_col].astype(str).str.strip().str.upper()
    df = df[df["STANCE"].isin(["POS", "NEG"])]
    df["Giorno"] = df["Data"].dt.date
    daily = df.groupby(["Giorno", "STANCE"]).size().unstack(fill_value=0)
    for c in ["POS", "NEG"]:
        if c not in daily.columns:
            daily[c] = 0
    daily["TOT"] = daily["POS"] + daily["NEG"]
    daily = daily[daily["TOT"] >= 1]
    daily.index = pd.to_datetime(daily.index)
    return daily["POS"] / daily["TOT"] * 100


# Tenta caricamento da master, altrimenti ricostruisce
if os.path.exists(MASTER_PATH):
    print(f"📂 Carico master da {MASTER_PATH}")
    master = pd.read_csv(MASTER_PATH, parse_dates=["Data"], index_col="Data")
else:
    print("⚠️ Master non trovato, ricostruisco da raw...")
    s_ref = serie_youtube(PATH_REFERENDUM, "Referendum")
    s_gov = serie_youtube(PATH_GOVERNO,    "Governo")
    s_sta = serie_stampa(PATH_STAMPA)
    data_start = max(s_ref.index.min(), s_gov.index.min(), s_sta.index.min())
    data_end   = min(s_ref.index.max(), s_gov.index.max(), s_sta.index.max())
    all_dates = pd.date_range(data_start, data_end, freq="D")
    master = pd.DataFrame(index=all_dates)
    master["Referendum"] = s_ref.reindex(all_dates)
    master["Governo"]    = s_gov.reindex(all_dates)
    master["Stampa"]     = s_sta.reindex(all_dates)
    master.index.name = "Data"

print(f"\n📐 Master: {len(master)} giorni × {len(master.columns)} serie")
print(f"   Periodo: {master.index.min():%d/%m/%Y} → {master.index.max():%d/%m/%Y}")
print(f"   Missing per serie:")
for c in master.columns:
    print(f"     {c}: {master[c].isna().sum()} NaN su {len(master)}")

# Strategia missing: interpolazione lineare con limite 3 giorni consecutivi
# (preserva la struttura temporale senza introdurre smoothing sistematico)
master_interp = master.interpolate(method="linear", limit=3).dropna()
print(f"\n📐 Dopo interpolazione lineare (limite 3gg): {len(master_interp)} osservazioni")

## §2 — Pre-test ADF di stazionarietà

Il test di Granger richiede serie **stazionarie**: media e varianza costanti nel tempo.
Su serie non-stazionarie i p-value diventano spurii — un risultato "significativo" potrebbe
riflettere solo una tendenza comune.

Verifica con il **test di Augmented Dickey-Fuller** (Dickey & Fuller 1979):
- H0: la serie ha una radice unitaria → non stazionaria
- p-value < 0,05 ⇒ rifiuto H0 ⇒ stazionaria

Se una serie risulta non-stazionaria, la differenzio (prima differenza). I test di Granger
sulle serie differenziate vanno interpretati come causalità sulle *variazioni*, non sui livelli.

**Atteso (Cap 6, §6.1):** tutte e tre le serie risultano già stazionarie nelle versioni grezze.

In [ ]:
from statsmodels.tsa.stattools import adfuller

print("=" * 70)
print("TEST ADF — STAZIONARIETÀ DELLE SERIE GIORNALIERE GREZZE")
print("=" * 70)
print(f"{'Serie':<15} {'ADF stat':>10} {'p-value':>10} {'Stazionaria?':>16}")
print("-" * 70)

stazionaria = {}
for col in master_interp.columns:
    s = master_interp[col].dropna()
    adf_stat, p_val, *_ = adfuller(s, autolag="AIC")
    is_staz = p_val < 0.05
    stazionaria[col] = is_staz
    flag = "✅ Sì" if is_staz else "❌ No → differenzio"
    print(f"{col:<15} {adf_stat:>10.3f} {p_val:>10.4f} {flag:>16}")

# Differenzia le serie non stazionarie
master_test = master_interp.copy()
differenziate = []
for col in master_interp.columns:
    if not stazionaria[col]:
        master_test[col] = master_interp[col].diff()
        differenziate.append(col)

master_test = master_test.dropna()
print(f"\n📐 Osservazioni finali per Granger: {len(master_test)}")

if differenziate:
    print(f"\n⚠️ Serie differenziate (Δ): {differenziate}")
    print("   I risultati Granger su queste serie si interpretano come")
    print("   causalità sulle *variazioni*, non sui livelli assoluti.")
else:
    print("\n✅ Nessuna serie differenziata: tutte stazionarie nei livelli.")

## §3 — Test di Granger su tutte le coppie ordinate

Per ciascuna coppia (causa → effetto) e ciascun lag in {3, 7}:
1. Costruisco un modello AR(p) per la serie *effetto* usando solo i suoi lag.
2. Aggiungo i lag della serie *causa* e ri-stimo il modello.
3. Test F congiunto sulla nullità dei coefficienti della causa (H0: causa non Granger-causa effetto).
4. Se p-value < 0,05, rifiuto H0: la causa Granger-causa l'effetto a quel lag.

Le 3 serie generano **6 coppie ordinate** (asimmetria fondamentale: A→B ≠ B→A).
Lag 3 e 7 motivati in §3.5.4: 3gg = reazione del dibattito online; 7gg = ciclo settimanale stampa.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

LAGS = [3, 7]
colonne = list(master_test.columns)
coppie = [(c, e) for c in colonne for e in colonne if c != e]


def granger_pair(data, cause_col, effect_col, lags):
    """Restituisce dict {lag: (F, p)} per la coppia ordinata."""
    pair = data[[effect_col, cause_col]].dropna()
    if len(pair) < max(lags) + 10:
        return {lag: (np.nan, np.nan) for lag in lags}
    try:
        res = grangercausalitytests(pair, maxlag=max(lags), verbose=False)
        return {lag: (res[lag][0]["ssr_ftest"][0], res[lag][0]["ssr_ftest"][1])
                for lag in lags}
    except Exception as e:
        print(f"  ⚠️ {cause_col} → {effect_col}: {e}")
        return {lag: (np.nan, np.nan) for lag in lags}


print("=" * 80)
print("GRANGER CAUSALITY — 6 coppie ordinate × 2 lag")
print("=" * 80)
print(f"{'Causa':<12} {'→':^3} {'Effetto':<12} {'Lag':>4} {'F':>9} {'p-value':>10}  Significativo?")
print("-" * 80)

results = []
for cause, effect in coppie:
    rs = granger_pair(master_test, cause, effect, LAGS)
    for lag, (f, p) in rs.items():
        sig = "✅ Sì" if (not np.isnan(p) and p < 0.05) else "❌ No"
        f_str   = f"{f:>9.3f}" if not np.isnan(f) else "       —"
        p_str   = f"{p:>10.4f}" if not np.isnan(p) else "         —"
        print(f"{cause:<12} → {effect:<12} {lag:>4} {f_str} {p_str}  {sig}")
        results.append({"causa": cause, "effetto": effect, "lag": lag,
                        "F": f, "p_value": p,
                        "significativo": (not np.isnan(p) and p < 0.05)})

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(WORKING_DIR, "granger_results.csv"),
                  index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("RELAZIONI SIGNIFICATIVE (p < 0,05)")
print("=" * 80)
sig = results_df[results_df["significativo"]].sort_values("p_value")
if len(sig) == 0:
    print("  Nessuna relazione significativa.")
else:
    for _, r in sig.iterrows():
        print(f"  • {r['causa']} → {r['effetto']}  "
              f"(lag {int(r['lag'])}gg | F={r['F']:.2f} | p={r['p_value']:.4f})")

## §4 — Heatmap dei p-value

Visualizzazione matriciale (causa sull'asse X, effetto sull'asse Y, valore = min p-value
sui due lag testati). Verde scuro = molto significativo; rosso = non significativo.

Il pattern atteso (Cap 6, §6.3):
- Colonne **Stampa** e **Governo** verso riga **Referendum** → verde (significative).
- Riga **Referendum** → tutte le colonne → rosso (non causa).
- Coppia **Stampa ↔ Governo** → rosso (non interagiscono direttamente).

In [ ]:
import plotly.graph_objects as go

# Matrice min p-value (causa × effetto)
serie_list = list(master_test.columns)
pivot_p = pd.DataFrame(index=serie_list, columns=serie_list, dtype=float)
pivot_lag = pd.DataFrame(index=serie_list, columns=serie_list, dtype=float)

for cause in serie_list:
    for effect in serie_list:
        if cause == effect:
            pivot_p.loc[effect, cause] = np.nan
            continue
        sub = results_df[(results_df["causa"] == cause) &
                         (results_df["effetto"] == effect)]
        if len(sub) > 0:
            best = sub.loc[sub["p_value"].idxmin()]
            pivot_p.loc[effect, cause] = best["p_value"]
            pivot_lag.loc[effect, cause] = best["lag"]

# Costruzione testo per la heatmap
text_matrix = []
for i, eff in enumerate(serie_list):
    row = []
    for j, cau in enumerate(serie_list):
        p = pivot_p.iloc[i, j]
        lag = pivot_lag.iloc[i, j]
        if np.isnan(p):
            row.append("—")
        else:
            row.append(f"p={p:.3f}<br>lag={int(lag)}")
    text_matrix.append(row)

fig_heat = go.Figure(data=go.Heatmap(
    z=pivot_p.values,
    x=pivot_p.columns.tolist(),
    y=pivot_p.index.tolist(),
    colorscale=[
        [0.00, "#15803d"],   # molto significativo
        [0.10, "#86efac"],
        [0.20, "#fef08a"],
        [0.50, "#fca5a5"],
        [1.00, "#991b1b"],   # non significativo
    ],
    zmin=0, zmax=0.5,
    text=text_matrix,
    texttemplate="%{text}",
    textfont=dict(size=14, color="black"),
    hovertemplate="Causa: %{x}<br>Effetto: %{y}<br>%{text}<extra></extra>",
    colorbar=dict(title="min p-value"),
))
fig_heat.update_layout(
    title="Granger Causality — p-value minimi su lag 3 e 7<br>"
          "<sub>Verde = significativo (p<0,05) | Asse X = causa | Asse Y = effetto</sub>",
    xaxis=dict(title="Causa", side="bottom"),
    yaxis=dict(title="Effetto"),
    plot_bgcolor="white",
    height=480, width=620,
    font=dict(size=13),
)
fig_heat.write_html(os.path.join(WORKING_DIR, "granger_heatmap.html"))
fig_heat.show(renderer="iframe_connected")
print("💾 Heatmap salvata: granger_heatmap.html")

## §5 — Schema causale asimmetrico

Network diagram delle relazioni significative emerse. L'idea: tradurre la matrice
in una rappresentazione visiva del flusso di influenza.

Configurazione attesa dalla tesi:
```
   Stampa ─── lag 3-7gg ───→ Referendum ←─── lag 3gg ─── Governo
```

**Stampa**: upstream puro (esce, non entra).
**Governo**: co-upstream (esce, non entra).
**Referendum YouTube**: downstream (entra, non esce).

In [ ]:
import plotly.graph_objects as go

# Posizionamento dei nodi nello spazio (manuale per chiarezza)
positions = {
    "Stampa":     (0.15, 0.5),
    "Governo":    (0.85, 0.5),
    "Referendum": (0.5,  0.15),
}
node_color = {
    "Stampa":     "#f59e0b",
    "Governo":    "#8b5cf6",
    "Referendum": "#3b82f6",
}

fig_net = go.Figure()

# Disegna le frecce significative
for _, r in sig.iterrows():
    x0, y0 = positions[r["causa"]]
    x1, y1 = positions[r["effetto"]]
    # spessore proporzionale a -log10(p)
    weight = max(2, min(8, -np.log10(r["p_value"]) * 3))
    fig_net.add_annotation(
        x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=3, arrowsize=1.5,
        arrowwidth=weight, arrowcolor="#10b981", opacity=0.85,
    )
    # label sulla freccia
    fig_net.add_annotation(
        x=(x0 + x1) / 2, y=(y0 + y1) / 2 + 0.04,
        text=f"<b>{r['causa']}→{r['effetto']}</b><br>"
             f"lag {int(r['lag'])}gg, p={r['p_value']:.4f}",
        showarrow=False, font=dict(size=11, color="#065f46"),
        bgcolor="rgba(255,255,255,0.85)", borderpad=4,
    )

# Disegna i nodi
for name, (x, y) in positions.items():
    fig_net.add_trace(go.Scatter(
        x=[x], y=[y], mode="markers+text",
        marker=dict(size=80, color=node_color[name],
                    line=dict(color="white", width=3)),
        text=[name], textposition="middle center",
        textfont=dict(size=14, color="white", family="Arial Black"),
        showlegend=False, hoverinfo="text", hovertext=f"<b>{name}</b>",
    ))

fig_net.update_layout(
    title="Schema causale di Granger — referendum 2026<br>"
          "<sub>Frecce = relazioni significative (p<0,05) | "
          "spessore ∝ −log₁₀(p)</sub>",
    xaxis=dict(visible=False, range=[-0.05, 1.05]),
    yaxis=dict(visible=False, range=[0, 1]),
    plot_bgcolor="white",
    height=480, width=720,
    margin=dict(l=20, r=20, t=80, b=20),
)
fig_net.write_html(os.path.join(WORKING_DIR, "granger_network.html"))
fig_net.show(renderer="iframe_connected")
print("💾 Network salvato: granger_network.html")

## §6 — Correlazione Pearson — controllo aggiuntivo

La correlazione contemporanea non è equivalente alla causalità di Granger:
- Una correlazione alta + Granger significativo ⇒ effetto temporale documentato.
- Una correlazione alta senza Granger ⇒ probabile fattore latente comune.
- Una correlazione bassa con Granger ⇒ effetto piccolo ma stabile nel tempo.

Calcolo Pearson tra le tre coppie. È un controllo pulito che non sostituisce il test
di Granger ma lo completa epistemologicamente.

In [ ]:
from scipy import stats

print("=" * 70)
print("CORRELAZIONE DI PEARSON — controllo (non causalità, solo associazione)")
print("=" * 70)
print(f"{'Coppia':<30} {'r':>8} {'p-value':>10}  Forza")
print("-" * 70)

for c1, c2 in [("Referendum", "Stampa"),
               ("Referendum", "Governo"),
               ("Stampa",     "Governo")]:
    s1 = master_test[c1].dropna()
    s2 = master_test[c2].dropna()
    idx = s1.index.intersection(s2.index)
    if len(idx) < 10:
        print(f"{c1} ↔ {c2:<15}  insuff. osserv.")
        continue
    r, p = stats.pearsonr(s1.loc[idx], s2.loc[idx])
    if abs(r) > 0.7:
        forza = "FORTE"
    elif abs(r) > 0.4:
        forza = "moderata"
    elif abs(r) > 0.2:
        forza = "debole"
    else:
        forza = "trascurabile"
    sig_p = "*" if p < 0.05 else " "
    print(f"{c1} ↔ {c2:<15}   r = {r:+5.3f}{sig_p:<2} {p:>9.4f}   {forza}")
print("\n* = p-value < 0.05")

## §7 — Verifica formale dell'ipotesi H3

H3 (Cap 1):
> «Esiste un'asimmetria causale di Granger tra stampa tradizionale e stance YouTube:
> la prima Granger-causa la seconda con un lag positivo, mentre il rapporto inverso
> non è statisticamente significativo. Una relazione analoga, con lag più breve, lega
> il consenso al governo alla stance sul referendum.»

Verifica strutturata: per ciascuna delle 4 condizioni di H3 segnalo se è soddisfatta.

In [ ]:
def check(causa, effetto, lag_atteso, deve_essere_significativo):
    """Verifica se una specifica relazione direzionale è (o non è) significativa al lag atteso."""
    sub = results_df[(results_df["causa"] == causa) &
                     (results_df["effetto"] == effetto) &
                     (results_df["lag"] == lag_atteso)]
    if len(sub) == 0:
        return None, None, None
    row = sub.iloc[0]
    p = row["p_value"]
    is_sig = (not np.isnan(p)) and p < 0.05
    soddisfatta = (is_sig == deve_essere_significativo)
    return p, is_sig, soddisfatta


condizioni_H3 = [
    ("Stampa", "Referendum", 3, True,
     "Stampa → Referendum a lag 3 deve essere significativo"),
    ("Stampa", "Referendum", 7, True,
     "Stampa → Referendum a lag 7 deve essere significativo"),
    ("Governo", "Referendum", 3, True,
     "Governo → Referendum a lag 3 deve essere significativo"),
    ("Referendum", "Stampa", 3, False,
     "Referendum → Stampa a lag 3 NON deve essere significativo"),
    ("Referendum", "Stampa", 7, False,
     "Referendum → Stampa a lag 7 NON deve essere significativo"),
    ("Referendum", "Governo", 3, False,
     "Referendum → Governo a lag 3 NON deve essere significativo"),
]

print("=" * 95)
print("VERIFICA FORMALE H3")
print("=" * 95)
n_ok = 0
for causa, effetto, lag, atteso, descrizione in condizioni_H3:
    p, is_sig, ok = check(causa, effetto, lag, atteso)
    if p is None:
        flag = "⚠️ Mancante"
        risultato = "—"
    else:
        flag = "✅ Confermata" if ok else "❌ Falsificata"
        risultato = f"p={p:.4f} ({'sig.' if is_sig else 'n.s.'})"
        if ok:
            n_ok += 1
    print(f"  {flag:<14} {descrizione}")
    print(f"                 → {risultato}")

totale = len(condizioni_H3)
print("\n" + "=" * 95)
if n_ok == totale:
    print(f"📊 H3 CONFERMATA INTEGRALMENTE: {n_ok}/{totale} condizioni soddisfatte")
elif n_ok >= totale - 1:
    print(f"📊 H3 SOSTANZIALMENTE CONFERMATA: {n_ok}/{totale} condizioni soddisfatte")
else:
    print(f"📊 H3 PARZIALMENTE CONFERMATA: {n_ok}/{totale} condizioni soddisfatte")
print("=" * 95)

## §8 — Sintesi epistemologica: termometro vs termostato

Lettura sintetica del Capitolo 6: l'asimmetria causale autorizza l'interpretazione
di YouTube come **termometro** dell'opinione pubblica (riceve influenza dagli altri due
attori) e **non come termostato** (non ne esercita in ritorno).

Tre cautele esplicite (Cap 6, §6.4.4):
1. Granger ≠ causalità in senso filosofico/sociologico ampio. Un fattore latente comune
   con sfasamenti differenti riprodurrebbe lo stesso pattern.
2. Risultato specifico al referendum 2026 e alle 3 serie misurate, non una proprietà
   generale di YouTube.
3. Non esclude canali di influenza non temporali (consolidamento di quadri interpretativi),
   che agirebbero senza produrre Granger immediato.

Pur con queste cautele, l'asimmetria pulita del sistema autorizza la lettura termometrica.

In [ ]:
# Calcolo dei flussi entranti / uscenti per ciascuna serie
flussi = {col: {"in": [], "out": []} for col in master_test.columns}
for _, r in sig.iterrows():
    flussi[r["effetto"]]["in"].append((r["causa"], r["lag"], r["p_value"]))
    flussi[r["causa"]]["out"].append((r["effetto"], r["lag"], r["p_value"]))

print("=" * 75)
print("DIAGNOSTICA FLUSSI — riepilogo per nodo")
print("=" * 75)
for nodo, fl in flussi.items():
    n_in  = len(fl["in"])
    n_out = len(fl["out"])
    if n_in > 0 and n_out == 0:
        ruolo = "🌡️ TERMOMETRO (riceve, non esercita)"
    elif n_out > 0 and n_in == 0:
        ruolo = "🎛️ TERMOSTATO (esercita, non riceve)"
    elif n_in > 0 and n_out > 0:
        ruolo = "🔀 INTERMEDIO (riceve e esercita)"
    else:
        ruolo = "🔘 ISOLATO (né in né out)"
    print(f"\n  {nodo}:  IN={n_in}, OUT={n_out}  →  {ruolo}")
    if fl["in"]:
        print(f"    Riceve da:")
        for cau, lag, p in fl["in"]:
            print(f"       ← {cau} (lag {int(lag)}, p={p:.4f})")
    if fl["out"]:
        print(f"    Esercita su:")
        for eff, lag, p in fl["out"]:
            print(f"       → {eff} (lag {int(lag)}, p={p:.4f})")

# Sintesi finale
print("\n" + "=" * 75)
print("INTERPRETAZIONE EPISTEMOLOGICA — termometro vs termostato")
print("=" * 75)

ref_in  = len(flussi["Referendum"]["in"])
ref_out = len(flussi["Referendum"]["out"])

if ref_in > 0 and ref_out == 0:
    print("""
✅ Lettura termometrica AUTORIZZATA:

   YouTube/Referendum riceve influenza dagli altri due attori (Stampa, Governo)
   ma non ne esercita in ritorno nei lag testati. Si comporta da sensore della
   temperatura del dibattito pubblico, non da regolatore.

   La pulizia dell'asimmetria — non un singolo coefficiente isolato, ma l'intera
   struttura coerente del sistema a tre serie e sei coppie — supera la soglia
   minima per autorizzare l'interpretazione termometrica.

   Resta valido l'avvertimento: è un risultato di causalità di Granger, non di
   causalità in senso filosofico ampio.
""")
else:
    print(f"""
⚠️ Lettura termometrica NON immediata: Referendum ha {ref_in} relazioni in entrata
   e {ref_out} in uscita.

   Lo schema osservato non riproduce in modo pulito la configurazione attesa
   dalla tesi. Riesaminare i lag e l'eventuale presenza di outlier nelle serie.
""")

## Sintesi del notebook

| Test / Risultato | Atteso (Cap 6) | Osservato (riempire dopo run) |
|---|---|---|
| ADF Referendum | p < 0,05 (stazionaria) | _da inserire_ |
| ADF Stampa | p < 0,05 (stazionaria) | _da inserire_ |
| ADF Governo | p < 0,05 (stazionaria) | _da inserire_ |
| Stampa → Ref lag 3 | p ≈ 0,037 | _da inserire_ |
| Stampa → Ref lag 7 | p ≈ 0,0029 | _da inserire_ |
| Governo → Ref lag 3 | p ≈ 0,038 | _da inserire_ |
| Ref → Stampa | n.s. | _da inserire_ |
| Ref → Governo | n.s. | _da inserire_ |
| Stampa ↔ Governo | n.s. | _da inserire_ |
| H3 (4 condizioni) | tutte confermate | _da inserire_ |
| YouTube come termometro | sì | _da inserire_ |

**File salvati in `/kaggle/working/`:**
- `granger_results.csv` — tabella completa di tutti i test
- `granger_heatmap.html` — heatmap matrice causa × effetto
- `granger_network.html` — network diagram dello schema causale

**Chiusura della pipeline analitica:** notebook 01-05 coprono raccolta → training → predizione/corpus → turning points → Granger. La tesi può ora essere ricomposta integralmente dai notebook (riproducibilità completa).